## Bước 10: Benchmark Evaluation
### 10.1 Đọc dữ liệu và tổng hợp kết quả đã có

## Thiết lập khả năng tái lập

Seed được đặt đồng nhất cho Python, NumPy và PyTorch. Các script mới còn sử dụng generator có seed cho DataLoader.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import set_global_seed

SEED = 42
set_global_seed(SEED)
print(f"Đã thiết lập seed tái lập: {SEED}")


In [1]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/depression_severity_preprocessed.csv')
label_map = {'minimum': 0, 'mild': 1, 'moderate': 2, 'severe': 3}
df['label_num'] = df['label'].map(label_map)

X_text = df['text_classical'].values
X_neural_text = df['text_neural'].values
y = df['label_num'].values

print("Số dòng:", len(df))

# Tổng hợp kết quả đã có từ các bước trước (Macro-F1, QWK, Accuracy)
# Với LogReg/SVM/XGBoost: dùng kết quả SAU khi tune (Bước 9)
# Với BiLSTM/BERT: dùng kết quả gốc (không tune, do giới hạn thời gian)
results_summary = {
    'Logistic Regression': {'accuracy': None, 'macro_f1': 0.4601, 'qwk': None},
    'SVM':                  {'accuracy': None, 'macro_f1': 0.4580, 'qwk': None},
    'XGBoost':              {'accuracy': None, 'macro_f1': 0.4475, 'qwk': None},
    'BiLSTM':               {'accuracy': 0.5988, 'macro_f1': 0.3279, 'qwk': 0.2464},
    'DistilBERT':           {'accuracy': 0.7159, 'macro_f1': 0.5518, 'qwk': 0.4956},
}

print("\nTổng hợp Macro-F1 hiện có:")
for model, res in results_summary.items():
    print(f"  {model}: {res['macro_f1']}")

Số dòng: 3519

Tổng hợp Macro-F1 hiện có:
  Logistic Regression: 0.4601
  SVM: 0.458
  XGBoost: 0.4475
  BiLSTM: 0.3279
  DistilBERT: 0.5518


### 10.2 Tính đầy đủ Accuracy/Macro-F1/QWK cho 3 model đã tune (qua 5-Fold)

In [2]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
import re

def extract_handcrafted_features(text):
    text_lower = text.lower()
    words = text_lower.split()
    n_words = len(words) if len(words) > 0 else 1
    features = {}
    features['word_count'] = len(words)
    first_person = ['i', 'me', 'my', 'mine', 'myself']
    features['first_person_ratio'] = sum(words.count(w) for w in first_person) / n_words
    negation = ['not', 'no', 'never', "n't", 'nothing', 'none']
    features['negation_ratio'] = sum(words.count(w) for w in negation) / n_words
    absolutist = ['always', 'never', 'everyone', 'nobody', 'everything', 'nothing',
                   'completely', 'totally', 'entirely']
    features['absolutist_ratio'] = sum(words.count(w) for w in absolutist) / n_words
    features['sentence_count'] = len(re.findall(r'[.!?]+', text))
    features['exclamation_ratio'] = text.count('!') / max(len(text), 1)
    warning_words = ['suicide', 'suicidal', 'kill', 'die', 'death', 'worthless', 'hopeless']
    features['warning_word_count'] = sum(text_lower.count(w) for w in warning_words)
    return features

handcrafted_all = df['text'].apply(extract_handcrafted_features).apply(pd.Series).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

final_results = {}

# ==== Logistic Regression (C=0.1) ====
fold_res = []
train_times = []
for train_idx, test_idx in skf.split(X_text, y):
    vec = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.9)
    Xtr = vec.fit_transform(X_text[train_idx]); Xte = vec.transform(X_text[test_idx])
    model = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=42)
    t0 = time.time(); model.fit(Xtr, y[train_idx]); train_times.append(time.time()-t0)
    pred = model.predict(Xte)
    fold_res.append({'acc': accuracy_score(y[test_idx], pred),
                      'f1': f1_score(y[test_idx], pred, average='macro'),
                      'qwk': cohen_kappa_score(y[test_idx], pred, weights='quadratic')})
fold_df = pd.DataFrame(fold_res)
final_results['Logistic Regression'] = {
    'accuracy': fold_df['acc'].mean(), 'macro_f1': fold_df['f1'].mean(), 'qwk': fold_df['qwk'].mean(),
    'train_time': np.mean(train_times)
}
print("Logistic Regression xong:", final_results['Logistic Regression'])

Logistic Regression xong: {'accuracy': np.float64(0.6504683660933661), 'macro_f1': np.float64(0.44584350430788283), 'qwk': np.float64(0.3529523412920522), 'train_time': np.float64(0.09789814949035644)}


### 10.3 SVM (C=0.5)

In [3]:
fold_res = []
train_times = []
for train_idx, test_idx in skf.split(X_text, y):
    vec = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.9)
    Xtr = vec.fit_transform(X_text[train_idx]); Xte = vec.transform(X_text[test_idx])
    model = SVC(C=0.5, kernel='linear', class_weight='balanced', random_state=42)
    t0 = time.time(); model.fit(Xtr, y[train_idx]); train_times.append(time.time()-t0)
    pred = model.predict(Xte)
    fold_res.append({'acc': accuracy_score(y[test_idx], pred),
                      'f1': f1_score(y[test_idx], pred, average='macro'),
                      'qwk': cohen_kappa_score(y[test_idx], pred, weights='quadratic')})
fold_df = pd.DataFrame(fold_res)
final_results['SVM'] = {
    'accuracy': fold_df['acc'].mean(), 'macro_f1': fold_df['f1'].mean(), 'qwk': fold_df['qwk'].mean(),
    'train_time': np.mean(train_times)
}
print("SVM xong:", final_results['SVM'])

SVM xong: {'accuracy': np.float64(0.6510337191258244), 'macro_f1': np.float64(0.4350587765721049), 'qwk': np.float64(0.3423234738354804), 'train_time': np.float64(10.012560653686524)}


### 10.4 XGBoost (tham số đã tune)

In [4]:
fold_res = []
train_times = []
for train_idx, test_idx in skf.split(X_text, y):
    vec = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.9)
    Xtr_tfidf = vec.fit_transform(X_text[train_idx]); Xte_tfidf = vec.transform(X_text[test_idx])

    svd = TruncatedSVD(n_components=300, random_state=42)
    Xtr_svd = svd.fit_transform(Xtr_tfidf); Xte_svd = svd.transform(Xte_tfidf)

    Xtr = np.hstack([Xtr_svd, handcrafted_all[train_idx]])
    Xte = np.hstack([Xte_svd, handcrafted_all[test_idx]])

    weights = compute_sample_weight('balanced', y[train_idx])

    model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                            random_state=42, eval_metric='mlogloss', n_jobs=1)
    t0 = time.time()
    model.fit(Xtr, y[train_idx], sample_weight=weights)
    train_times.append(time.time()-t0)

    pred = model.predict(Xte)
    fold_res.append({'acc': accuracy_score(y[test_idx], pred),
                      'f1': f1_score(y[test_idx], pred, average='macro'),
                      'qwk': cohen_kappa_score(y[test_idx], pred, weights='quadratic')})
    print(f"  Fold xong, mất {train_times[-1]:.1f}s")

fold_df = pd.DataFrame(fold_res)
final_results['XGBoost'] = {
    'accuracy': fold_df['acc'].mean(), 'macro_f1': fold_df['f1'].mean(), 'qwk': fold_df['qwk'].mean(),
    'train_time': np.mean(train_times)
}
print("\nXGBoost xong:", final_results['XGBoost'])

  Fold xong, mất 12.1s
  Fold xong, mất 11.3s
  Fold xong, mất 11.8s
  Fold xong, mất 12.3s
  Fold xong, mất 12.4s

XGBoost xong: {'accuracy': np.float64(0.6697962466054571), 'macro_f1': np.float64(0.4017774711172919), 'qwk': np.float64(0.3424000722482546), 'train_time': np.float64(11.969584608078003)}


### 10.5 Bảng Benchmark hoàn chỉnh (5 model)

In [5]:
final_results['BiLSTM'] = {'accuracy': 0.5988, 'macro_f1': 0.3279, 'qwk': 0.2464, 'train_time': 90.0}
final_results['DistilBERT'] = {'accuracy': 0.7159, 'macro_f1': 0.5518, 'qwk': 0.4956, 'train_time': 2913.0}

benchmark_df = pd.DataFrame(final_results).T
benchmark_df = benchmark_df.round(4)
benchmark_df = benchmark_df[['accuracy', 'macro_f1', 'qwk', 'train_time']]
benchmark_df.columns = ['Accuracy', 'Macro-F1', 'QWK', 'Training Time (s)']

print(benchmark_df)

benchmark_df.to_csv('../outputs/reports/benchmark_5models.csv')
print("\nĐã lưu: outputs/reports/benchmark_5models.csv")

                     Accuracy  Macro-F1     QWK  Training Time (s)
Logistic Regression    0.6505    0.4458  0.3530             0.0979
SVM                    0.6510    0.4351  0.3423            10.0126
XGBoost                0.6698    0.4018  0.3424            11.9696
BiLSTM                 0.5988    0.3279  0.2464            90.0000
DistilBERT             0.7159    0.5518  0.4956          2913.0000

Đã lưu: outputs/reports/benchmark_5models.csv


### 10.6 Đo Inference Time (thời gian dự đoán)

In [6]:
# Lấy 1 câu mẫu để đo thời gian dự đoán
sample_text_classical = X_text[0]
sample_text_neural = X_neural_text[0]

inference_times = {}

# ==== Logistic Regression ====
vec = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=2, max_df=0.9)
Xtr = vec.fit_transform(X_text)
model_lr = LogisticRegression(C=0.1, max_iter=1000, class_weight='balanced', random_state=42)
model_lr.fit(Xtr, y)

t0 = time.time()
_ = model_lr.predict(vec.transform([sample_text_classical]))
inference_times['Logistic Regression'] = (time.time() - t0) * 1000  # ms

# ==== SVM ====
model_svm = SVC(C=0.5, kernel='linear', class_weight='balanced', random_state=42)
model_svm.fit(Xtr, y)

t0 = time.time()
_ = model_svm.predict(vec.transform([sample_text_classical]))
inference_times['SVM'] = (time.time() - t0) * 1000

print("Đo xong Logistic Regression và SVM:")
print(inference_times)

Đo xong Logistic Regression và SVM:
{'Logistic Regression': 0.7755756378173828, 'SVM': 4.096269607543945}


### 10.7 Inference Time — XGBoost

In [7]:
svd_full = TruncatedSVD(n_components=300, random_state=42)
Xtr_svd_full = svd_full.fit_transform(Xtr)
Xtr_xgb_full = np.hstack([Xtr_svd_full, handcrafted_all])

weights_full = compute_sample_weight('balanced', y)
model_xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                            random_state=42, eval_metric='mlogloss', n_jobs=1)
model_xgb.fit(Xtr_xgb_full, y, sample_weight=weights_full)

sample_hc = handcrafted_all[0:1]
sample_tfidf = vec.transform([sample_text_classical])
sample_svd = svd_full.transform(sample_tfidf)
sample_xgb_input = np.hstack([sample_svd, sample_hc])

t0 = time.time()
_ = model_xgb.predict(sample_xgb_input)
inference_times['XGBoost'] = (time.time() - t0) * 1000

print("Đo xong XGBoost:", inference_times['XGBoost'], "ms")

Đo xong XGBoost: 1.4967918395996094 ms


### 10.8 Inference Time — BiLSTM (chỉ cần kiến trúc, không cần train lại)

In [8]:
import torch
import torch.nn as nn
from collections import Counter

MAX_LEN = 150

def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)
    return text.split()

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=64, num_classes=4, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        final_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.dropout(final_hidden)
        out = self.fc(out)
        return out

# Xây vocab nhanh (không cần train model, chỉ cần kích thước đúng)
counter = Counter()
for text in X_neural_text[:500]:  # chỉ dùng 500 dòng để xây vocab nhanh, đủ để có kích thước thực tế
    counter.update(simple_tokenize(text))
vocab_size_estimate = len(counter) + 2

model_bilstm_test = BiLSTMClassifier(vocab_size=vocab_size_estimate)
model_bilstm_test.eval()

sample_ids = torch.randint(0, vocab_size_estimate, (1, MAX_LEN))

t0 = time.time()
with torch.no_grad():
    _ = model_bilstm_test(sample_ids)
inference_times['BiLSTM'] = (time.time() - t0) * 1000

print("Đo xong BiLSTM:", inference_times['BiLSTM'], "ms")

Đo xong BiLSTM: 7.235050201416016 ms


### 10.9 Inference Time — DistilBERT (load model đã lưu)

In [9]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_bert_loaded = AutoModelForSequenceClassification.from_pretrained('../models/distilbert_model')
tokenizer_loaded = AutoTokenizer.from_pretrained('../models/distilbert_model')
model_bert_loaded.eval()

sample_encoding = tokenizer_loaded(sample_text_neural, padding='max_length', truncation=True,
                                     max_length=150, return_tensors='pt')

t0 = time.time()
with torch.no_grad():
    _ = model_bert_loaded(**sample_encoding)
inference_times['DistilBERT'] = (time.time() - t0) * 1000

print("Đo xong DistilBERT:", inference_times['DistilBERT'], "ms")
print("\n=== TỔNG HỢP INFERENCE TIME (ms) ===")
for model, t in inference_times.items():
    print(f"  {model}: {t:.2f} ms")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7434.30it/s]


Đo xong DistilBERT: 149.70874786376953 ms

=== TỔNG HỢP INFERENCE TIME (ms) ===
  Logistic Regression: 0.78 ms
  SVM: 4.10 ms
  XGBoost: 1.50 ms
  BiLSTM: 7.24 ms
  DistilBERT: 149.71 ms


### 10.10 Lưu bảng Benchmark hoàn chỉnh

In [10]:
benchmark_df['Inference Time (ms)'] = [
    inference_times['Logistic Regression'],
    inference_times['SVM'],
    inference_times['XGBoost'],
    inference_times['BiLSTM'],
    inference_times['DistilBERT'],
]
benchmark_df = benchmark_df.round(4)

print(benchmark_df)

benchmark_df.to_csv('../outputs/reports/benchmark_5models_final.csv')
print("\nĐã lưu: outputs/reports/benchmark_5models_final.csv")

                     Accuracy  Macro-F1     QWK  Training Time (s)  \
Logistic Regression    0.6505    0.4458  0.3530             0.0979   
SVM                    0.6510    0.4351  0.3423            10.0126   
XGBoost                0.6698    0.4018  0.3424            11.9696   
BiLSTM                 0.5988    0.3279  0.2464            90.0000   
DistilBERT             0.7159    0.5518  0.4956          2913.0000   

                     Inference Time (ms)  
Logistic Regression               0.7756  
SVM                               4.0963  
XGBoost                           1.4968  
BiLSTM                            7.2351  
DistilBERT                      149.7087  

Đã lưu: outputs/reports/benchmark_5models_final.csv


## Benchmark thống nhất trên cùng các lần chia

Notebook cũ kết hợp kết quả 5-Fold của một số mô hình với một lần chia 80/20 của DistilBERT. Script mới đánh giá năm mô hình trên đúng cùng các seed và cùng train/test indices. Kết quả cuối được báo cáo bằng mean ± std, không nhập cứng.

In [ ]:
RUN_UNIFIED_BENCHMARK = False  # Chạy đủ 5 mô hình trên CPU có thể mất nhiều giờ

if RUN_UNIFIED_BENCHMARK:
    import subprocess
    import sys
    from pathlib import Path

    PROJECT_ROOT = Path('..').resolve()
    subprocess.run(
        [
            sys.executable,
            'src/run_unified_benchmark.py',
            '--seeds', '42', '52', '62',
            '--models', 'logreg', 'svm', 'xgboost', 'bilstm', 'distilbert',
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )
else:
    print('Đặt RUN_UNIFIED_BENCHMARK=True để chạy benchmark thống nhất.')


In [ ]:
from pathlib import Path
import pandas as pd

summary_path = Path('../outputs/reports/unified_repeated_holdout_summary.csv')
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary)
else:
    print('Chưa có kết quả benchmark thống nhất.')
